# University Program Skill Analysis with LAiSER

This notebook provides a quick overview of skills taught across various university programs by extracting and categorizing skills from their descriptions.

- Author: Satya Phanindra Kumar Kalaga
- Date: September 2025
- Last update: March 2026 - Luis Gonzalez

## 1. Setup

First, let's install the `laiser` package and other necessary libraries.

In [ ]:
!pip install laiser pandas matplotlib seaborn

## 2. Import Libraries

In [ ]:
import pandas as pd
from laiser.skill_extractor_refactored import SkillExtractorRefactored
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Initialize the Skill Extractor

We'll initialize the `SkillExtractorRefactored`. Remember to replace `"your_model_id"` and `"your_hf_token"` with your actual Hugging Face credentials.

In [ ]:
# Replace with your model ID and API key
# gemini = gemini-2.5-flash-lite
# Other options: Anthropic's Claude models, OpenAI's GPT-4, etc.


model_id = "gemini"
use_gpu = False   
api_key = "provide-your-api-key-here"
try:
    se = SkillExtractorRefactored(
        model_id=model_id,
        api_key=api_key,
        use_gpu=use_gpu,
    )
    print("Skill Extractor initialized successfully!")
except Exception as e:
    print(f"Error initializing Skill Extractor: {e}")

## 4. Load Program Description Data

We will load a sample dataset of university program descriptions. This dataset will serve as our source for skill extraction. For this example, we'll create a sample DataFrame.

In [ ]:
# Sample data representing university programs
data = {
    'program_id': [101, 102, 103, 104],
    'program_name': ['Computer Science', 'Data Science', 'Business Analytics', 'Mechanical Engineering'],
    'description': [
        'Our Computer Science program focuses on software development, algorithms, and data structures. Students will learn Python, Java, and C++.',
        'The Data Science program teaches students to analyze large datasets. Key topics include machine learning, statistical modeling, and data visualization using R and Python.',
        'The Business Analytics program bridges the gap between data and decision-making. Students learn SQL, Tableau, and business intelligence techniques.',
        'Our Mechanical Engineering program covers thermodynamics, fluid mechanics, and robotics. Students gain hands-on experience with CAD software and 3D printing.'
    ]
}
program_data = pd.DataFrame(data)
print("Program data loaded successfully:")
program_data

## 5. Extract Skills from Program Descriptions

Now we'll use the `extract_and_align` method to get the skills from the program descriptions.

We set a similarity threshold value of 0.5

Higher values = stricter matching, fewer results.

Lower values = more lenient matching, more results.

In [ ]:
try:
    extracted_skills = se.extract_and_align(
        program_data,
        id_column="program_id",
        text_columns=["description"],
        input_type='syllabus',  # Using 'syllabus' as the input type for program descriptions
        similarity_threshold=0.5
    )
    print("Skills extracted successfully!")
    print(extracted_skills.head())
except Exception as e:
    print(f"Error during skill extraction: {e}")

## 6. Categorize and Visualize Skills

To get a better overview, we'll merge the extracted skills with the program names and then visualize the distribution of skills across different programs.

In [ ]:
if 'extracted_skills' in locals() and not extracted_skills.empty:
    extracted_skills['Research ID'] = pd.to_numeric(
        extracted_skills['Research ID'],
        errors='coerce'
    )

    skills_with_programs = pd.merge(
        extracted_skills,
        program_data[['program_id', 'program_name']],
        left_on='Research ID',
        right_on='program_id'
    )

    plt.figure(figsize=(12, 7))
    sns.countplot(
        y='program_name',
        data=skills_with_programs,
        order=skills_with_programs['program_name'].value_counts().index
    )
    plt.title('Number of Skills Mentioned per Program')
    plt.xlabel('Number of Skills')
    plt.ylabel('Program')
    plt.tight_layout()
    plt.show()

    for program in skills_with_programs['program_name'].unique():
        print(f"--- Skills for {program} ---")
        skills_list = skills_with_programs[
            skills_with_programs['program_name'] == program
        ]['Taxonomy Skill'].dropna().tolist()
        print(", ".join(skills_list))
        print("\n")
else:
    print("No skills were extracted to visualize.")